# Prediction Pipeline for Bike-Sharing Demand

In real-world machine learning systems, models are not used in isolation. Instead, they are embedded within a prediction pipeline that transforms raw input data into meaningful predictions.

This notebook implements the prediction pipeline by:

- Taking weather forecast data obtained from an API
- Engineering features required by the model
- Applying the trained regression model
- Generating predicted bike-sharing demand
- Categorizing predictions into demand levels

This pipeline mirrors the logic implemented in the Shiny dashboard backend and ensures consistency between model development and deployment.

By structuring the prediction workflow in this manner, we create a reusable and scalable system for real-time inference.

## Import Required Libraries

We import libraries required for:

- Data manipulation and transformation
- Loading the trained model
- Numerical computations
- Datetime processing for feature extraction

In [ ]:
# Import pandas for structured data manipulation
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Import joblib to load saved machine learning model
import joblib

# Import datetime module for handling date and time features
from datetime import datetime

## Load API Data

We load the weather dataset generated from the API integration stage.

This dataset contains:

- Temperature
- Humidity
- Wind speed
- Weather condition
- Datetime information

These variables will be transformed into model-ready features.

In [ ]:
# Load weather data (output from Notebook 09)
weather_df = pd.read_csv("weather_data.csv")

# Display first few rows to inspect data
weather_df.head()

## Feature Engineering: Datetime Processing

Datetime information is critical for capturing temporal patterns in bike demand.

We extract:

- Hour of the day (important for commuting patterns)
- Month (used for season classification)

These features align with the model inputs used during training.

In [ ]:
# Convert datetime column from string to pandas datetime format
weather_df["datetime"] = pd.to_datetime(weather_df["datetime"])

# Extract hour from datetime (range: 0–23)
weather_df["hour"] = weather_df["datetime"].dt.hour

# Extract month from datetime (used to derive season)
weather_df["month"] = weather_df["datetime"].dt.month

# Display updated DataFrame
weather_df.head()

## Feature Engineering: Season Mapping

Seasonal patterns significantly influence bike-sharing demand.

We map each month to a corresponding season:

- Winter: December to February
- Spring: March to May
- Summer: June to August
- Autumn: September to November

This categorical feature will be used in prediction.

In [ ]:
# Define function to map month to season
def get_season(month):
    
    # Winter months
    if month in [12, 1, 2]:
        return "WINTER"
    
    # Spring months
    elif month in [3, 4, 5]:
        return "SPRING"
    
    # Summer months
    elif month in [6, 7, 8]:
        return "SUMMER"
    
    # Autumn months
    else:
        return "AUTUMN"

# Apply function to create season column
weather_df["season"] = weather_df["month"].apply(get_season)

# Display updated data
weather_df.head()

## Load Trained Model

We load the final trained model saved in the previous notebook.

This model will be used to generate predictions based on engineered features.

In [ ]:
# Load trained model from file
model = joblib.load("final_bike_demand_model.pkl")

## Generate Predictions

We select the required features and apply the model to generate predicted bike demand.

The features used must match those used during model training.

In [ ]:
# Select relevant features for prediction
# Ensure these match training features exactly

feature_cols = ["temperature", "humidity", "wind_speed", "hour"]

# Extract feature matrix
X_pred = weather_df[feature_cols]

# Generate predictions using trained model
weather_df["bike_prediction"] = model.predict(X_pred)

# Display results
weather_df.head()

## Post-processing Predictions

Predicted values may sometimes be negative due to model behavior.

Since bike demand cannot be negative, we adjust values:

- Replace negative predictions with zero

In [ ]:
# Replace negative predictions with zero
weather_df["bike_prediction"] = weather_df["bike_prediction"].apply(
    lambda x: max(0, x)
)

## Categorize Demand Levels

To improve interpretability, we categorize predictions into levels:

- Small demand
- Medium demand
- Large demand

This is useful for visualization in the dashboard.

In [ ]:
# Define function to categorize demand levels
def demand_level(pred):
    
    # Low demand
    if pred <= 1000:
        return "small"
    
    # Medium demand
    elif pred < 3000:
        return "medium"
    
    # High demand
    else:
        return "large"

# Apply categorization
weather_df["demand_level"] = weather_df["bike_prediction"].apply(demand_level)

# Display updated data
weather_df.head()

## Summary

In this notebook, we implemented a complete prediction pipeline:

- Loaded real-time weather data
- Engineered temporal and seasonal features
- Applied trained regression model
- Generated predictions
- Categorized demand levels

This pipeline directly supports the Shiny dashboard by providing structured, prediction-ready data.

---

## Author & Acknowledgment

**Author:**  
Deepan Mehta  

**GitHub Profile:**  
https://github.com/deepan-mehta-analytics

This notebook implements the prediction pipeline by transforming API-derived weather data into model-ready features and generating bike-sharing demand predictions.

The structure and methodology are inspired by IBM Skills Network instructional labs on deploying machine learning models and building inference pipelines.

Special acknowledgment is given to:

- Yan Luo  
- Jeff Grossman  

---

## Project Context

This notebook represents the real-time inference stage within the end-to-end data science pipeline:

- Data Collection  
- Data Wrangling (ETL)  
- Exploratory Data Analysis (EDA)  
- Model Development  
- Model Evaluation  
- API Integration  
- **Prediction Pipeline (Real-Time Inference)**  

---

## Notes

All code and explanations have been independently structured to reflect a production-oriented workflow, ensuring alignment between model development and deployment in the Shiny dashboard.

---